# readme-drift on real repositories

How `readme-drift` behaves on popular open-source projects: where it helps, where it raises false alarms, and where it misses.
Everything below is computed from the runner's output (`run_benchmark.py`), so the numbers can be regenerated.

**Four measurements per repository**

1. `--verify` on the current README: how many commands are verified, broken or unverifiable, and which "broken" ones are real.
2. **History replay:** the last 150 code-changing commits, each staged the way `git add` would and checked by the hook's code path.
3. **Injected drift:** rename or break things the README mentions, and see what is caught.
4. **The real pre-commit hook** (`pre-commit try-repo`) on a few of those commits.

**Reading the labels.** A finding is *useful* if a maintainer would plausibly want the warning, and a *false alarm* otherwise.
Labels come from `triage.py`: deterministic rules over evidence the runner records (for example, whether a "missing" file exists next to the README),
plus a short list of manual judgements. Anything the rules cannot decide is shown as *unlabeled*, never silently counted.

In [1]:
import os
import sys
from collections import Counter
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE if (HERE / "triage.py").exists() else HERE / "benchmarks"))
import triage
from triage import label_check, label_finding, load, md_table, metrics, pct, show

RESULTS = Path(os.environ.get("BENCH_RESULTS", HERE / "results"))
variants = {}
if RESULTS.is_dir():
    variants = {d.name: load(d) for d in sorted(RESULTS.iterdir()) if d.is_dir() and any(d.glob("*.json"))}
BASE = "baseline" if "baseline" in variants else next(iter(variants), None)
records = variants.get(BASE, [])
print(f"results: {RESULTS}")
print(f"variants: {list(variants) or 'none - run the benchmark first (see the last section)'}  | analysing: {BASE}")

results: /tmp/claude-1000/-home-sachin-nandakumar-projects-readme-drift/f6cd0d98-dce2-4229-a033-3bb0eaa8f70f/scratchpad/bench/rt2
variants: ['baseline']  | analysing: baseline


## 1 · What was run

In [3]:
rows = [
    [r["repo"], r.get("head", "?"), len(r.get("readmes", [])), f"{r['tool']['branch']}@{r['tool']['commit']}"]
    for r in records
]
show(md_table(["repository", "HEAD", "READMEs found", "tool version"], rows))

| repository | HEAD | READMEs found | tool version |
|---|---|---|---|
| Textualize/rich | 3a109cc | 5 | feature/benchmark-popular-repos@1212632 |
| encode/httpx | 7cff5f7 | 1 | feature/benchmark-popular-repos@1212632 |
| expressjs/express | 43fca52 | 2 | feature/benchmark-popular-repos@1212632 |
| fastapi/fastapi | b283119 | 1 | feature/benchmark-popular-repos@1212632 |
| fastapi/full-stack-fastapi-template | 77e550a | 4 | feature/benchmark-popular-repos@1212632 |
| pallets/flask | 49943b5 | 5 | feature/benchmark-popular-repos@1212632 |
| psf/requests | 611c616 | 4 | feature/benchmark-popular-repos@1212632 |
| pydantic/pydantic | e56f60f | 12 | feature/benchmark-popular-repos@1212632 |
| pytest-dev/pytest | 36eed6b | 4 | feature/benchmark-popular-repos@1212632 |
| python-poetry/poetry | 8c3ddc3 | 27 | feature/benchmark-popular-repos@1212632 |

## 2 · `--verify`: commands in the README

Each command is verified, broken or unverifiable. "Broken" is the only outcome that fails a run, so it is the one that has to be right.

In [5]:
rows, broken_rows, unverifiable = [], [], Counter()
for r in records:
    checks = r.get("verify", {}).get("checks", [])
    count = Counter(c["status"] for c in checks)
    rows.append([r["repo"], count["verified"], count["broken"], count["unverifiable"]])
    for c in checks:
        if c["status"] == "unverifiable":
            unverifiable[c["reason"][:64]] += 1
        elif c["status"] == "broken":
            label, cause = label_check(r["repo"], c)
            broken_rows.append([r["repo"], f"{c['file']}:{c['line']}", f"`{c['raw']}`", c["reason"], label, cause])
show(md_table(["repository", "verified", "broken", "unverifiable"], rows))
useful = sum(1 for b in broken_rows if b[4] == "useful")
show(f"**Precision of 'broken': {pct(useful, len(broken_rows))}**")
show(md_table(["repository", "where", "command", "reported as", "judged", "cause"], broken_rows))

| repository | verified | broken | unverifiable |
|---|---|---|---|
| Textualize/rich | 3 | 2 | 1 |
| encode/httpx | 0 | 0 | 4 |
| expressjs/express | 0 | 0 | 12 |
| fastapi/fastapi | 0 | 0 | 3 |
| fastapi/full-stack-fastapi-template | 14 | 3 | 14 |
| pallets/flask | 0 | 1 | 8 |
| psf/requests | 0 | 2 | 3 |
| pydantic/pydantic | 8 | 3 | 6 |
| pytest-dev/pytest | 0 | 0 | 0 |
| python-poetry/poetry | 0 | 0 | 0 |

**Precision of 'broken': 0% (0/11)**

| repository | where | command | reported as | judged | cause |
|---|---|---|---|---|---|
| Textualize/rich | examples/README.md:3 | `python example.py` | path 'example.py' does not exist | false-alarm | illustrative-command |
| Textualize/rich | examples/README.md:3 | `python justify.py` | path 'justify.py' does not exist | false-alarm | nested-readme-resolution |
| fastapi/full-stack-fastapi-template | frontend/README.md:20 | `bash scripts/prestart.sh` | path 'scripts/prestart.sh' does not exist | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | backend/README.md:15 | `docker compose up -d db mailpit` | service 'mailpit' not found in compose.yml | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | backend/README.md:22 | `bash scripts/prestart.sh` | path 'scripts/prestart.sh' does not exist | false-alarm | nested-readme-resolution |
| pallets/flask | examples/celery/README.md:22 | `flask -A task_app run --debug` | 'flask' has no option --debug | false-alarm | flag-on-subcommand-not-readable |
| psf/requests | tests/certs/expired/README.md:9 | `make clean` | make target 'clean' not found in Makefile | false-alarm | nested-readme-resolution |
| psf/requests | tests/certs/expired/README.md:10 | `make all` | make target 'all' not found in Makefile | false-alarm | nested-readme-resolution |
| pydantic/pydantic | pydantic-core/README.md:104 | `make build-dev` | make target 'build-dev' not found in Makefile | false-alarm | nested-readme-resolution |
| pydantic/pydantic | pydantic-core/README.md:105 | `make build-prod` | make target 'build-prod' not found in Makefile | false-alarm | nested-readme-resolution |
| pydantic/pydantic | pydantic-core/README.md:123 | `make build-profiling` | make target 'build-profiling' not found in Makefile | false-alarm | nested-readme-resolution |

In [6]:
show("**Why commands could not be verified** (this is coverage, not error):")
show(md_table(["reason", "count"], unverifiable.most_common(10)))

**Why commands could not be verified** (this is coverage, not error):

| reason | count |
|---|---|
| not a script invocation | 7 |
| no verifier for 'pip' | 6 |
| no verifier for 'bun' | 5 |
| no verifier for 'git' | 4 |
| no verifier for 'cd' | 3 |
| no verifier for 'alembic' | 3 |
| 'pip' is not a module of this repository | 2 |
| no verifier for 'uv' | 2 |
| could not read the options of 'fastapi' statically | 2 |
| no verifier for 'bunx' | 2 |

## 3 · The hook on real history

Each of the last 150 code-changing commits is staged on top of its parent, exactly as `git add` would, and checked.
A finding here means the hook would have interrupted that commit.

In [8]:
rows, listing, causes = [], [], Counter()
for r in records:
    commits = [c for c in r.get("replay", {}).get("commits", []) if "error" not in c]
    flagged = [c for c in commits if c["findings"]]
    findings = [(c, f) for c in commits for f in c["findings"]]
    rows.append([r["repo"], len(commits), len(flagged), len(findings)])
    for c, f in findings:
        label, cause = label_finding(r["repo"], f)
        causes[(label, cause)] += 1
        m = f["matches"][0] if f["matches"] else {"text": "", "line": "?", "readme": "?"}
        listing.append([r["repo"], c["sha"][:7], f"`{f['symbol']}`", f["type"], f"{m['readme']}:{m['line']}", m["text"][:60].replace("`", "'"), label, cause])
show(md_table(["repository", "commits replayed", "commits flagged", "findings"], rows))
useful = sum(1 for x in listing if x[6] == "useful")
show(f"**Precision: {pct(useful, len(listing))}** of {len(listing)} findings were useful.")
show(md_table(["judged", "cause", "count"], [[k[0], k[1], v] for k, v in causes.most_common()]))

| repository | commits replayed | commits flagged | findings |
|---|---|---|---|
| Textualize/rich | 150 | 3 | 5 |
| encode/httpx | 132 | 4 | 9 |
| expressjs/express | 150 | 2 | 2 |
| fastapi/fastapi | 80 | 2 | 15 |
| fastapi/full-stack-fastapi-template | 150 | 14 | 46 |
| pallets/flask | 150 | 6 | 9 |
| psf/requests | 150 | 1 | 1 |
| pydantic/pydantic | 150 | 5 | 31 |
| pytest-dev/pytest | 150 | 0 | 0 |
| python-poetry/poetry | 150 | 2 | 3 |

**Precision: 0% (0/121)** of 121 findings were useful.

| judged | cause | count |
|---|---|---|
| false-alarm | config-key-in-prose | 71 |
| unlabeled | needs review | 27 |
| false-alarm | common-word-in-prose | 15 |
| false-alarm | config-name-survives-migration | 8 |

In [9]:
show("**Every finding, judged:**")
show(md_table(["repository", "commit", "symbol", "change", "README match", "matched text", "judged", "cause"], listing[:60]))
if len(listing) > 60:
    show(f"... and {len(listing) - 60} more.")

**Every finding, judged:**

| repository | commit | symbol | change | README match | matched text | judged | cause |
|---|---|---|---|---|---|---|---|
| Textualize/rich | 69536ec | `python` | removed | README.md:51 | python -m pip install rich | false-alarm | config-key-in-prose |
| Textualize/rich | 69536ec | `include` | removed | README.md:272 | The columns may be configured to show any details you want.  | false-alarm | config-key-in-prose |
| Textualize/rich | 69536ec | `pygments` | removed | README.md:382 | Rich uses the [pygments](https://pygments.org/) library to i | false-alarm | config-name-survives-migration |
| Textualize/rich | e8e797d | `install` | signature_changed | README.md:51 | python -m pip install rich | false-alarm | common-word-in-prose |
| Textualize/rich | 041406c | `asv` | removed | benchmarks/README.md:7 | The benchmarks use a tool called [Airspeed Velocity](https:/ | unlabeled | needs review |
| encode/httpx | 1805ee0 | `head` | signature_changed | README.md:40 | '<!doctype html>\n<html>\n<head>\n<title>Example Domain</tit | false-alarm | common-word-in-prose |
| encode/httpx | 1805ee0 | `request` | signature_changed | README.md:55 | Sending a request... | false-alarm | common-word-in-prose |
| encode/httpx | 8e36f2b | `head` | signature_changed | README.md:40 | '<!doctype html>\n<html>\n<head>\n<title>Example Domain</tit | false-alarm | common-word-in-prose |
| encode/httpx | 8e36f2b | `request` | signature_changed | README.md:55 | Sending a request... | false-alarm | common-word-in-prose |
| encode/httpx | 8cd952c | `Advanced Usage` | removed | README.md:114 | For more advanced topics, see the [Advanced Usage](https://w | false-alarm | config-key-in-prose |
| encode/httpx | 8cd952c | `Usage` | removed | README.md:114 | For more advanced topics, see the [Advanced Usage](https://w | false-alarm | config-key-in-prose |
| encode/httpx | 8cd952c | `features` | removed | README.md:74 | Plus all the standard features of 'requests'... | false-alarm | config-key-in-prose |
| encode/httpx | f8981f3 | `head` | signature_changed | README.md:40 | '<!doctype html>\n<html>\n<head>\n<title>Example Domain</tit | false-alarm | common-word-in-prose |
| encode/httpx | f8981f3 | `request` | signature_changed | README.md:55 | Sending a request... | false-alarm | common-word-in-prose |
| expressjs/express | 6c98f80 | `include` | removed | benchmarks/README.md:30 | You can use 'make && node -v' to include the node.js version | false-alarm | config-key-in-prose |
| expressjs/express | 088856c | `include` | removed | benchmarks/README.md:30 | You can use 'make && node -v' to include the node.js version | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `main` | removed | README.md:149 | Create a file 'main.py' with: | unlabeled | needs review |
| fastapi/fastapi | 8972a2d | `with` | removed | README.md:30 | FastAPI is a modern, fast (high-performance), web framework  | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `name` | removed | README.md:388 | * Check that it has a required attribute 'name' that should  | unlabeled | needs review |
| fastapi/fastapi | 8972a2d | `uses` | removed | README.md:170 | If your code uses 'async' / 'await', use 'async def': | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `steps` | removed | README.md:143 | If you prefer to use 'pip', install 'fastapi[standard]' insi | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `with` | removed | README.md:30 | FastAPI is a modern, fast (high-performance), web framework  | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `name` | removed | README.md:388 | * Check that it has a required attribute 'name' that should  | unlabeled | needs review |
| fastapi/fastapi | 8972a2d | `uses` | removed | README.md:170 | If your code uses 'async' / 'await', use 'async def': | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `default` | removed | README.md:230 | By default, 'fastapi dev' will start with auto-reload enable | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `types` | removed | README.md:273 | Declare the body using standard Python types, thanks to Pyda | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `steps` | removed | README.md:143 | If you prefer to use 'pip', install 'fastapi[standard]' insi | false-alarm | config-key-in-prose |
| fastapi/fastapi | 8972a2d | `required` | removed | README.md:386 | * Without the 'None' it would be required (as is the body in | false-alarm | config-key-in-prose |
| fastapi/fastapi | 0a8c3fa | `Path` | signature_changed | README.md:361 | * Path parameters. | unlabeled | needs review |
| fastapi/fastapi | 0a8c3fa | `Cookie` | signature_changed | README.md:438 | * **Cookie Sessions** | unlabeled | needs review |
| fastapi/fastapi | 0a8c3fa | `Query` | signature_changed | README.md:362 | * Query parameters. | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | f8dd304 | `main` | removed | frontend/README.md:32 | * In the 'backend/app/main.py' file, remove the 'app.fronten | false-alarm | common-word-in-prose |
| fastapi/full-stack-fastapi-template | f8dd304 | `main` | removed | frontend/README.md:32 | * In the 'backend/app/main.py' file, remove the 'app.fronten | false-alarm | common-word-in-prose |
| fastapi/full-stack-fastapi-template | 7f11815 | `docs` | removed | README.md:11 | - 🔍 [Pydantic](https://docs.pydantic.dev), used by FastAPI,  | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 7f11815 | `steps` | removed | frontend/README.md:36 | * In the '.github/workflows/deploy.yml' file, remove the **S | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 7f11815 | `uses` | removed | frontend/README.md:72 | By default, the built frontend uses the same origin as the F | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `with` | removed | README.md:25 | - ✅ Tests with [Pytest](https://pytest.org). | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `uses` | removed | frontend/README.md:68 | By default, the built frontend uses the same origin as the F | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `SECRET_KEY` | removed | README.md:130 | - 'SECRET_KEY' | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `POSTGRES_PASSWORD` | removed | README.md:132 | - 'POSTGRES_PASSWORD' | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `steps` | removed | frontend/README.md:64 | Notice that everytime the backend changes (changing the Open | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `FIRST_SUPERUSER_PASSWORD` | removed | README.md:131 | - 'FIRST_SUPERUSER_PASSWORD' | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `release` | removed | README.md:170 | Check the file [release-notes.md](./release-notes.md). | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `FIRST_SUPERUSER` | removed | README.md:131 | - 'FIRST_SUPERUSER_PASSWORD' | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | 6b9fa41 | `environment` | removed | README.md:134 | You can (and should) pass these as environment variables fro | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 6dbb2e8 | `recommended` | removed | frontend/README.md:7 | - [Bun](https://bun.sh/) (recommended) or [Node.js](https:// | false-alarm | config-name-survives-migration |
| fastapi/full-stack-fastapi-template | 32ab6dd | `POSTGRES_PASSWORD` | removed | README.md:132 | - 'POSTGRES_PASSWORD' | false-alarm | config-name-survives-migration |
| fastapi/full-stack-fastapi-template | af79180 | `default` | removed | README.md:21 | - 🔒 Secure password hashing by default. | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 9d54108 | `prestart` | removed | frontend/README.md:18 | Run 'uv run bash scripts/prestart.sh' and 'uv run fastapi de | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 9d54108 | `prestart` | removed | frontend/README.md:18 | Run 'uv run bash scripts/prestart.sh' and 'uv run fastapi de | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 9d54108 | `prestart` | removed | frontend/README.md:18 | Run 'uv run bash scripts/prestart.sh' and 'uv run fastapi de | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 23b0ba9 | `with` | removed | README.md:25 | - ✅ Tests with [Pytest](https://pytest.org). | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 23b0ba9 | `needs` | removed | backend/README.md:138 | Values coming from the backend are declared as Jinja placeho | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 9c8983e | `FRONTEND_HOST` | removed | frontend/README.md:38 | If you want, you can also remove the 'FRONTEND_HOST' environ | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | 9c8983e | `FRONTEND_HOST` | removed | frontend/README.md:38 | If you want, you can also remove the 'FRONTEND_HOST' environ | unlabeled | needs review |
| fastapi/full-stack-fastapi-template | 0137c3f | `restart` | removed | backend/README.md:46 | There is also a command override that runs 'fastapi run --re | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 0137c3f | `traefik` | removed | README.md:26 | - 📞 [Traefik](https://traefik.io) as a reverse proxy / load  | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 0137c3f | `command` | removed | README.md:142 | You have to change them with a secret key, to generate secre | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 0137c3f | `image` | removed | README.md:14 | - 🧩 Built into the backend image and served by FastAPI on th | false-alarm | config-key-in-prose |
| fastapi/full-stack-fastapi-template | 0137c3f | `restart` | removed | backend/README.md:46 | There is also a command override that runs 'fastapi run --re | false-alarm | config-key-in-prose |

... and 61 more.

## 4 · What if `docs/` were scanned too?

The same commits, with every Markdown and reStructuredText file under `docs/` added to the READMEs. This is a what-if, not current behaviour.

In [11]:
docs_names, readme_hits, rows, causes = 0, 0, [], Counter()
for r in records:
    commits = [c for c in r.get("replay_docs", {}).get("commits", []) if "error" not in c]
    findings = [(c, f) for c in commits for f in c["findings"]]
    for _, f in findings:
        causes[label_finding(r["repo"], f)] += 1
    rows.append([r["repo"], r.get("replay_docs", {}).get("docs_files", 0), len([c for c in commits if c["findings"]]), len(findings)])
show(md_table(["repository", "docs files scanned", "commits flagged", "findings"], rows))
useful = sum(v for k, v in causes.items() if k[0] == "useful")
show(f"**Precision with docs scanned: {pct(useful, sum(causes.values()))}**")
show(md_table(["judged", "cause", "count"], [[k[0], k[1], v] for k, v in causes.most_common()]))

| repository | docs files scanned | commits flagged | findings |
|---|---|---|---|
| Textualize/rich | 63 | 4 | 10 |
| encode/httpx | 23 | 4 | 28 |
| expressjs/express | 0 | 0 | 0 |
| fastapi/fastapi | 158 | 6 | 48 |
| fastapi/full-stack-fastapi-template | 0 | 0 | 0 |
| pallets/flask | 76 | 15 | 37 |
| psf/requests | 15 | 1 | 1 |
| pydantic/pydantic | 91 | 10 | 288 |
| pytest-dev/pytest | 0 | 0 | 0 |
| python-poetry/poetry | 16 | 3 | 17 |

**Precision with docs scanned: 0% (4/429)**

| judged | cause | count |
|---|---|---|
| unlabeled | needs review | 287 |
| false-alarm | config-key-in-prose | 81 |
| false-alarm | common-word-in-prose | 26 |
| false-alarm | auto-generated-directive | 14 |
| false-alarm | config-name-survives-migration | 13 |
| false-alarm | backtick-prefix | 4 |
| useful | documented public API changed signature | 4 |

## 5 · Injected drift: what gets caught?

For names the README mentions, the runner breaks the repository (rename, new required parameter, removed Makefile target, npm script, compose service, script file),
stages the change, and checks whether the diff check or `--verify` notices. This measures recall on cases with a known right answer.

In [13]:
by_kind, misses = {}, []
for r in records:
    for i in r.get("inject", {}).get("injections", []):
        row = by_kind.setdefault(i["kind"], [0, 0, 0, 0])
        row[0] += 1
        row[1] += bool(i["detected"])
        row[2] += bool(i.get("verify_detected"))
        row[3] += bool(i["detected"] or i.get("verify_detected"))
        if not (i["detected"] or i.get("verify_detected")):
            why = [w for w, on in (("noise word", i["noise_word"]), ("short name", i["short"])) if on]
            if i.get("styles"):
                why.append("README style: " + "/".join(i["styles"]))
            misses.append([r["repo"], i["kind"], f"`{i.get('symbol', i['name'])}`", ", ".join(why) or "-"])
rows = [[k, v[0], pct(v[1], v[0]), pct(v[2], v[0]), pct(v[3], v[0])] for k, v in sorted(by_kind.items())]
total = sum(v[0] for v in by_kind.values())
show(md_table(["injected drift", "injected", "diff check caught", "--verify caught", "either"], rows))
show(f"**Overall: {pct(sum(v[3] for v in by_kind.values()), total)} caught.**")
show("**What was missed:**")
show(md_table(["repository", "kind", "symbol", "why"], misses))

| injected drift | injected | diff check caught | --verify caught | either |
|---|---|---|---|---|
| compose-service-removed | 2 | 50% (1/2) | 50% (1/2) | 50% (1/2) |
| makefile-target-removed | 6 | 83% (5/6) | 66% (4/6) | 100% (6/6) |
| python-rename | 11 | 45% (5/11) | 0% (0/11) | 45% (5/11) |
| python-signature | 11 | 27% (3/11) | 0% (0/11) | 27% (3/11) |
| script-file-renamed | 6 | 0% (0/6) | 100% (6/6) | 100% (6/6) |

**Overall: 58% (21/36) caught.**

**What was missed:**

| repository | kind | symbol | why |
|---|---|---|---|
| Textualize/rich | python-rename | `Console.print` | README style: bare |
| Textualize/rich | python-rename | `Layout.tree` | README style: bare |
| Textualize/rich | python-signature | `Console.print` | README style: bare |
| Textualize/rich | python-signature | `Layout.tree` | README style: bare |
| Textualize/rich | python-signature | `Markdown` | README style: bare |
| fastapi/fastapi | python-rename | `RouteContext.name` | noise word, README style: bare |
| fastapi/fastapi | python-signature | `UJSONResponse` | README style: bare |
| fastapi/fastapi | python-signature | `RouteContext.name` | noise word, README style: bare |
| fastapi/full-stack-fastapi-template | compose-service-removed | `db` | short name |
| psf/requests | python-rename | `Response.json` | README style: bare |
| psf/requests | python-signature | `Response.json` | README style: bare |
| pytest-dev/pytest | python-rename | `ExceptionInfo.type` | noise word |
| pytest-dev/pytest | python-rename | `Argument.type` | noise word |
| pytest-dev/pytest | python-signature | `ExceptionInfo.type` | noise word |
| pytest-dev/pytest | python-signature | `Argument.type` | noise word |

In [14]:
styles = {}
for r in records:
    for i in r.get("inject", {}).get("injections", []):
        if i["kind"] in ("python-rename", "python-signature") and i.get("styles"):
            for s in i["styles"]:
                row = styles.setdefault(s, [0, 0])
                row[0] += 1
                row[1] += bool(i["detected"])
show("**Python references by how the README writes them:**")
show(md_table(["README style", "injected", "caught"], [[k, v[0], pct(v[1], v[0])] for k, v in sorted(styles.items())]))

**Python references by how the README writes them:**

| README style | injected | caught |
|---|---|---|
| bare | 18 | 44% (8/18) |

## 6 · The real pre-commit hook

`pre-commit try-repo` runs the hook definition itself (`.pre-commit-hooks.yaml`) on staged real commits, which also checks the file-type filter and the install.

In [16]:
runs = [(r["repo"], run) for r in records for run in r.get("hook", {}).get("runs", [])]
status = Counter(run["status"] for _, run in runs)
show(md_table(["outcome", "runs"], status.most_common()))
seconds = sorted(run["seconds"] for _, run in runs if run["status"] in ("passed", "failed"))
if seconds:
    show(f"Whole `pre-commit` invocation: median {seconds[len(seconds)//2]} s (includes git and framework start-up).")
errors = [(repo, run) for repo, run in runs if run["status"].startswith("error")]
show(md_table(["repository", "commit", "tail of output"], [[r, x["sha"], x["tail"][-90:].replace("\n", " ")] for r, x in errors[:5]]))

| outcome | runs |
|---|---|
| passed | 40 |

Whole `pre-commit` invocation: median 4.18 s (includes git and framework start-up).

| repository | commit | tail of output |
|---|---|---|

## 7 · Speed

In [18]:
m = metrics(records)
verify_secs = [r["verify"]["seconds"] for r in records if "seconds" in r.get("verify", {})]
show(md_table(
    ["measure", "value"],
    [
        ["diff check, median", f"{m.get('diff_seconds_median', 'n/a')} s"],
        ["diff check, 95th percentile", f"{m.get('diff_seconds_p95', 'n/a')} s"],
        ["diff check, slowest", f"{m.get('diff_seconds_max', 'n/a')} s"],
        ["--verify, slowest repository", f"{max(verify_secs) if verify_secs else 'n/a'} s"],
    ],
))

| measure | value |
|---|---|
| diff check, median | 0.043 s |
| diff check, 95th percentile | 0.221 s |
| diff check, slowest | 73.477 s |
| --verify, slowest repository | 0.085 s |

## 8 · Comparing tool versions

When more than one results folder exists (for example `baseline` and a branch with fixes), this table shows each headline number side by side.

In [20]:
KEYS = [
    ("verify: broken reported", "verify_broken"),
    ("verify: broken that were useful", "verify_broken_useful"),
    ("verify: broken that were false alarms", "verify_broken_false-alarm"),
    ("diff: findings", "diff_findings"),
    ("diff: useful", "diff_findings_useful"),
    ("diff: false alarms", "diff_findings_false-alarm"),
    ("diff: unlabeled", "diff_findings_unlabeled"),
    ("docs what-if: findings", "docs_findings"),
    ("docs what-if: useful", "docs_findings_useful"),
    ("injected drift caught", "inject_caught"),
    ("injected drift total", "inject_total"),
]
table = {name: metrics(recs) for name, recs in variants.items()}
rows = [[label] + [table[n].get(key, 0) for n in table] for label, key in KEYS]
show(md_table(["measure"] + list(table), rows) if table else "No results yet.")

| measure | baseline |
|---|---|
| verify: broken reported | 11 |
| verify: broken that were useful | 0 |
| verify: broken that were false alarms | 9 |
| diff: findings | 121 |
| diff: useful | 0 |
| diff: false alarms | 94 |
| diff: unlabeled | 27 |
| docs what-if: findings | 429 |
| docs what-if: useful | 4 |
| injected drift caught | 21 |
| injected drift total | 36 |

## 9 · How to run it

```bash
# 1. clone each repository, run every phase, write one JSON file per repository
poetry run python benchmarks/run_benchmark.py \
    --workdir /tmp/bench/repos --results benchmarks/results/baseline \
    --hook-repo /path/to/clean/readme-drift/checkout --precommit-path /path/to/pre_commit

# 2. measure another version of the tool by pointing PYTHONPATH at its checkout
PYTHONPATH=/path/to/other/checkout poetry run python benchmarks/run_benchmark.py \
    --workdir /tmp/bench/repos --results benchmarks/results/fixed --phases verify replay inject replay_docs

# 3. open this notebook; it reads every folder under benchmarks/results/
```

Cloned repositories are untrusted input: the tool only parses them statically and never runs their code.